# 12 — Sprint 4: fundamentos da análise comercial

Este notebook concentra somente o vocabulário técnico compartilhado, a normalização não destrutiva e o acesso à base de conhecimento. Ele não executa uma análise sozinho; prepara as funções pequenas que os notebooks seguintes reutilizam no mesmo kernel.

## Contrato comum

A transcrição original nunca é modificada. Uma cópia normalizada é usada apenas para comparação lexical. Os modos aceitos são `auto`, `full` e `fallback`, e todas as dependências pesadas serão carregadas sob demanda nos módulos especializados.

In [ ]:
from __future__ import annotations
from collections import Counter
from functools import lru_cache
import json
import math
from pathlib import Path
import re
from typing import Any
import unicodedata

SUPPORTED_MODES = {"auto", "full", "fallback"}

PORTUGUESE_STOPWORDS = {
    "a", "ao", "aos", "as", "com", "como", "da", "das",
    "de", "do", "dos", "e", "em", "entre", "essa", "esse",
    "esta", "este", "foi", "isso", "mais", "mas", "muito",
    "na", "nao", "nas", "no", "nos", "o", "os", "ou",
    "para", "pela", "pelo", "por", "que", "se", "sem", "ser",
    "sua", "sao", "tem", "um", "uma", "voce", "cliente",
    "reuniao", "atual", "cenario", "acompanhamento",
}


## Preparação interna do texto

Os chunks por palavras limitam o tamanho enviado aos modelos. A normalização remove diferenças de caixa e acentuação apenas da cópia de trabalho; `_tokens` também descarta stopwords para o ranking lexical.

In [ ]:
def _word_chunks(text: str, max_words: int = 80) -> list[str]:
    words = text.split()
    return [" ".join(words[start:start + max_words]) for start in range(0, len(words), max_words)]

def _normalize(text: str) -> str:
    normalized = unicodedata.normalize("NFKD", str(text).casefold())
    return "".join(char for char in normalized if not unicodedata.combining(char))

def _tokens(text: str) -> list[str]:
    return [
        token
        for token in re.findall(r"[a-z0-9+]+", _normalize(text))
        if len(token) >= 2 and token not in PORTUGUESE_STOPWORDS
    ]


## Localização dos artefatos

A raiz é descoberta a partir do diretório atual ou de `PROJECT_ROOT`, usado pelos testes. O catálogo TOTVS e seus aliases são lidos uma vez e mantidos em cache durante a sessão.

In [ ]:
def _project_root() -> Path:
    configured = globals().get("PROJECT_ROOT")
    if configured is not None:
        root = Path(configured).resolve()
        if (root / "data" / "knowledge_base").exists():
            return root
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data" / "knowledge_base").exists():
            return candidate
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto Wedjat.")

def _load_catalog() -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    root = _project_root()
    knowledge_base = json.loads(
        (root / "data" / "knowledge_base" / "totvs_rag_kb_v1.json").read_text(
            encoding="utf-8"
        )
    )
    aliases_payload = json.loads(
        (root / "data" / "knowledge_base" / "rag_aliases.json").read_text(
            encoding="utf-8"
        )
    )
    return knowledge_base, aliases_payload["groups"]
